# Image Segmentation — Exercises

Companion to the note [Image Segmentation](Image%20Segmentation.md).

Practise the segmentation task types, U-Net design, IoU/Dice and the arithmetic of transposed and dilated convolutions by hand, then implement per-class IoU and mIoU, a confusion-matrix evaluator, soft Dice loss, transposed convolution, connected-component labelling (semantic → instances) and bilinear upsampling from scratch in NumPy.

**13 exercises** · 🧠 Concept ×3 · ✍️ By hand ×4 · 💻 Code ×6

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper first; then type your numbers into the cell to check them.
- 💻 *Code* exercises: replace `return None` / `None` with your implementation and run the cell; the checks print ✅, ❌ or ⏳ (not attempted).
- Every exercise has a folded 💡 **Hint** and ✅ **Solution**. Try for a few minutes before opening them.

In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))
rng = np.random.default_rng(7)


## Part A · Concepts

### Exercise 1 · Semantic, instance or panoptic?
*🧠 Concept · ★☆☆*

For each application choose semantic, instance or panoptic segmentation and say why: (a) counting cells in a microscope image where cells touch, (b) marking drivable road for a self-driving car, (c) a street scene where you need both the sky/road and each individual pedestrian, (d) blurring every person's face separately in a crowd photo.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Do you need to *separate* objects of the same class? Do you need to label amorphous 'stuff' like sky or road?

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) **Instance**: touching cells of one class would merge into one region in a semantic mask, so you couldn't count them. (b) **Semantic**: road is 'stuff'; you only need which pixels are road. (c) **Panoptic**: stuff (sky, road) semantically + things (pedestrians) as instances. (d) **Instance**: each face needs its own mask to be processed separately.

</details>

### Exercise 2 · Why U-Net works with few images
*🧠 Concept · ★★☆*

U-Net (2015) was designed for biomedical images with only ~30 labelled training images. Explain (i) what the encoder and decoder each contribute, (ii) what the skip connections carry and why they are concatenated rather than added, and (iii) why a fully convolutional design helps when data is scarce.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Think about *what* vs *where*, about what information is lost by pooling, and about how many training examples one image provides for a per-pixel loss.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(i) The encoder downsamples and builds semantic features (*what*); the decoder upsamples back to full resolution (*where*). (ii) Skip connections carry high-resolution encoder feature maps (edges, fine detail) that pooling destroyed; **concatenation** keeps them as separate channels so the decoder can learn how to combine them with the coarse semantic features (addition would force both into the same feature space). (iii) Every pixel is a training target, so one image gives thousands of (correlated) examples; a fully convolutional network shares weights across positions, needs fewer parameters than dense layers, and works on any image size. Heavy augmentation (elastic deformations) did the rest.

</details>

### Exercise 3 · The 95 % accuracy trap
*🧠 Concept · ★★☆*

A tumour segmentation model reports 97 % pixel accuracy. Tumour pixels make up 3 % of each image. What might the model be doing? Name two metrics and one loss that would expose or fix the problem.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

What does a model that predicts 'background' everywhere score?

</details>

<details>
<summary><b>✅ Solution</b></summary>

Predicting 'background' for every pixel already gives **97 % pixel accuracy**, so the number says nothing. Use **IoU/mIoU** or **Dice** per class (the tumour class would get 0), and train with **Dice loss** (or focal / class-weighted cross-entropy, or CE + Dice) so the rare class carries weight in the loss.

</details>

## Part B · By hand

### Exercise 4 · IoU and Dice from pixel counts
*✍️ By hand · ★☆☆*

A predicted mask covers 120 pixels, the ground truth 150 pixels, and they overlap in 90 pixels. Compute the IoU and the Dice coefficient, and verify $\text{Dice}=\frac{2\,\text{IoU}}{1+\text{IoU}}$.

In [ ]:
iou_ex = None
dice_ex = None
check('IoU', iou_ex, 0.5); check('Dice', dice_ex, 2 / 3, tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

Union = |P| + |G| − |P ∩ G|.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Union $=120+150-90=180$, IoU $=90/180=0.5$. Dice $=2\cdot90/(120+150)=180/270=0.667$. Check: $2\cdot0.5/1.5=0.667$ ✓.

```python
iou_ex = 90 / 180
dice_ex = 2 * 90 / (120 + 150)
```

</details>

### Exercise 5 · Dice ↔ IoU
*✍️ By hand · ★☆☆*

(a) A model has IoU 0.8 on a mask. What is its Dice? (b) Another has Dice 0.5. What is its IoU? (c) Which number is always larger, and when are they equal?

In [ ]:
dice_a = None
iou_b = None
check('(a) Dice from IoU 0.8', dice_a, 1.6 / 1.8, tol=1e-4); check('(b) IoU from Dice 0.5', iou_b, 1 / 3, tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

Invert $D = 2J/(1+J)$ to get $J = D/(2-D)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $2\cdot0.8/1.8=0.889$. (b) $J=D/(2-D)=0.5/1.5=0.333$. (c) Dice ≥ IoU always; they are equal only at 0 and 1.

```python
dice_a = 2 * 0.8 / 1.8
iou_b = 0.5 / (2 - 0.5)
```

</details>

### Exercise 6 · Transposed convolution sizes
*✍️ By hand · ★★☆*

A decoder upsamples a $16\times16$ feature map with a transposed convolution: kernel $F=4$, stride $S=2$, padding $P=1$. What is the output size? Which kernel size would you need with $S=2, P=0$ to go from 16 to exactly 32?

In [ ]:
out_size = None
kernel_p0 = None
check('output size', out_size, 32); check('kernel with P=0', kernel_p0, 2)

<details>
<summary><b>💡 Hint</b></summary>

Output $=(W-1)S-2P+F$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$(16-1)\cdot2-2\cdot1+4=32$. With $P=0$: $(16-1)\cdot2+F=32 \Rightarrow F=2$ (a $2\times2$ kernel with stride 2, the standard 'up-convolution' of U-Net).

```python
out_size = (16 - 1) * 2 - 2 * 1 + 4
kernel_p0 = 32 - (16 - 1) * 2
```

</details>

### Exercise 7 · Receptive field of dilated convolutions
*✍️ By hand · ★★☆*

Three stacked $3\times3$ convolutions with dilations 1, 2 and 4 (stride 1). What is the receptive field (side length in pixels) of one output pixel? How many weights per input/output channel pair do the three layers use in total, and how big would a single ordinary convolution with the same receptive field be?

In [ ]:
rf = None
weights = None
check('receptive field', rf, 15); check('weights', weights, 27)

<details>
<summary><b>💡 Hint</b></summary>

A $k\times k$ conv with dilation $d$ adds $(k-1)d$ to the receptive field.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$1+2\cdot1+2\cdot2+2\cdot4=15$, so $15\times15$. Weights: $3\cdot9=27$. A single ordinary conv would need a $15\times15$ kernel = 225 weights. Dilations grow context exponentially without pooling, which is why DeepLab uses them to keep resolution.

```python
rf = 1 + 2 * (1 + 2 + 4)
weights = 3 * 9
```

</details>

## Part C · Code from scratch

### Exercise 8 · Per-class IoU and mIoU
*💻 Code · ★☆☆*

Implement `per_class_iou(pred, gt, n_classes)` for integer label maps of the same shape, returning an array of IoUs with `np.nan` for classes absent from both `pred` and `gt`, and `miou(pred, gt, n_classes)` = the mean over the non-NaN classes.

In [ ]:
def per_class_iou(pred, gt, n_classes):
    return None

def miou(pred, gt, n_classes):
    return None
gt_ = np.array([[0, 0, 1], [0, 1, 1], [2, 2, 2]]); pr_ = np.array([[0, 1, 1], [0, 1, 1], [2, 2, 0]])
pc_ = per_class_iou(pr_, gt_, 4)
check('per-class IoU (NaN shown as -1)', None if pc_ is None else np.nan_to_num(pc_, nan=-1), [2 / 4, 3 / 4, 2 / 3, -1])
check('mIoU ignores absent class 3', miou(pr_, gt_, 4), np.mean([0.5, 0.75, 2 / 3]))

<details>
<summary><b>💡 Hint</b></summary>

For class c: intersection = `((pred == c) & (gt == c)).sum()`, union = `((pred == c) | (gt == c)).sum()`. Use `np.nanmean`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Absent classes are excluded rather than counted as 1 or 0, which is the convention of most benchmarks (Pascal VOC, Cityscapes).

```python
def per_class_iou(pred, gt, n_classes):
    out = np.full(n_classes, np.nan)
    for c in range(n_classes):
        inter = np.sum((pred == c) & (gt == c)); union = np.sum((pred == c) | (gt == c))
        if union > 0:
            out[c] = inter / union
    return out

def miou(pred, gt, n_classes):
    return float(np.nanmean(per_class_iou(pred, gt, n_classes)))
```

</details>

### Exercise 9 · A confusion-matrix evaluator
*💻 Code · ★★☆*

Benchmarks accumulate one confusion matrix over the whole dataset instead of averaging per-image IoUs. Implement `confusion(pred, gt, n)` (rows = ground truth, columns = prediction) **without loops** using `np.bincount`, and `iou_from_confusion(cm)` that returns per-class IoU (NaN where the union is 0).

In [ ]:
def confusion(pred, gt, n):
    return None

def iou_from_confusion(cm):
    return None
cm_ = confusion(pr_, gt_, 4)
check('confusion matrix', cm_, [[2, 1, 0, 0], [0, 3, 0, 0], [1, 0, 2, 0], [0, 0, 0, 0]])
P = rng.integers(0, 5, (20, 30)); G = rng.integers(0, 5, (20, 30))
check('IoU from confusion = per-class IoU', None if cm_ is None else iou_from_confusion(confusion(P, G, 5)), per_class_iou(P, G, 5) if per_class_iou(P, G, 5) is not None else None)

<details>
<summary><b>💡 Hint</b></summary>

Encode each pixel pair as `n * gt + pred`, bincount with `minlength=n*n`, reshape. IoU = diagonal / (row sum + column sum − diagonal).

</details>

<details>
<summary><b>✅ Solution</b></summary>

Accumulating counts first (dataset-level IoU) weights large objects more than averaging per-image scores; both conventions exist, so always say which one you report.

```python
def confusion(pred, gt, n):
    return np.bincount(n * gt.ravel() + pred.ravel(), minlength=n * n).reshape(n, n)

def iou_from_confusion(cm):
    tp = np.diag(cm).astype(float); union = cm.sum(0) + cm.sum(1) - tp
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(union > 0, tp / union, np.nan)
```

</details>

### Exercise 10 · Soft Dice loss and its gradient
*💻 Code · ★★☆*

Implement `soft_dice_loss(p, g, eps=1e-6)` $=1-\frac{2\sum p g+\epsilon}{\sum p+\sum g+\epsilon}$ for a probability map `p` and binary mask `g`, and `soft_dice_grad(p, g, eps=1e-6)` = its gradient w.r.t. `p` (derive it with the quotient rule).

In [ ]:
def soft_dice_loss(p, g, eps=1e-6):
    return None

def soft_dice_grad(p, g, eps=1e-6):
    return None
g_ = (rng.random((8, 8)) < 0.2).astype(float); p_ = rng.random((8, 8))
check('perfect prediction → 0', soft_dice_loss(g_.copy(), g_), 0.0, tol=1e-5)
check('disjoint → ≈ 1', soft_dice_loss(1 - g_, g_), 1.0, tol=1e-5)
eps_ = 1e-6; num = np.zeros_like(p_)
if soft_dice_loss(p_, g_) is not None:
    for i in np.ndindex(p_.shape):
        q = p_.copy(); q[i] += eps_; a = soft_dice_loss(q, g_); q[i] -= 2 * eps_; num[i] = (a - soft_dice_loss(q, g_)) / (2 * eps_)
check('gradient vs finite differences', soft_dice_grad(p_, g_), num if soft_dice_loss(p_, g_) is not None else None, tol=1e-5)

<details>
<summary><b>💡 Hint</b></summary>

With $I=\sum pg$ and $S=\sum p+\sum g$: $\frac{\partial}{\partial p_i}\frac{2I+\epsilon}{S+\epsilon}=\frac{2g_i(S+\epsilon)-(2I+\epsilon)}{(S+\epsilon)^2}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The loss only depends on the overlap relative to the total foreground, so it doesn't care how many background pixels there are, which is exactly why it handles class imbalance.

```python
def soft_dice_loss(p, g, eps=1e-6):
    return 1 - (2 * np.sum(p * g) + eps) / (np.sum(p) + np.sum(g) + eps)

def soft_dice_grad(p, g, eps=1e-6):
    I, S = np.sum(p * g), np.sum(p) + np.sum(g)
    return -(2 * g * (S + eps) - (2 * I + eps)) / (S + eps) ** 2
```

</details>

### Exercise 11 · Transposed convolution by scattering
*💻 Code · ★★★*

Implement a single-channel `conv_transpose2d(x, k, stride)` with no padding: every input pixel `x[i, j]` adds `x[i, j] * k` into the output starting at `(i*stride, j*stride)`. Output size $(H-1)S+F$. Then explain the **checkerboard** pattern you get with `k = ones((3, 3))`, stride 2.

In [ ]:
def conv_transpose2d(x, k, stride):
    return None
out_ = conv_transpose2d(np.ones((3, 3)), np.ones((3, 3)), 2)
check('output shape', None if out_ is None else out_.shape, (7, 7))
check('checkerboard counts (row 1)', None if out_ is None else out_[1], [1, 1, 2, 1, 2, 1, 1])
check('centre gets 4 contributions', None if out_ is None else out_[2, 2], 4)
check('stride 2, kernel 2: no overlap', conv_transpose2d(np.array([[1., 2], [3, 4]]), np.ones((2, 2)), 2), np.kron([[1, 2], [3, 4]], np.ones((2, 2))))

<details>
<summary><b>💡 Hint</b></summary>

Allocate `np.zeros(((H-1)*s + F, (W-1)*s + F))` and add the kernel block per input pixel.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With kernel 3 and stride 2 the scattered blocks overlap unevenly: some output pixels receive contributions from 1, some from 2, some from 4 inputs, hence the checkerboard. Use kernel size divisible by the stride (e.g. 4 with stride 2), or bilinear upsampling followed by a normal convolution.

```python
def conv_transpose2d(x, k, stride):
    H, W = x.shape; F = k.shape[0]
    out = np.zeros(((H - 1) * stride + F, (W - 1) * stride + F))
    for i in range(H):
        for j in range(W):
            out[i * stride:i * stride + F, j * stride:j * stride + F] += x[i, j] * k
    return out
```

</details>

### Exercise 12 · From semantic to instances: connected components
*💻 Code · ★★☆*

Semantic segmentation can't count touching objects, but separate blobs can be counted. Implement `label_components(mask)` for a binary mask: return an integer array where each 4-connected foreground region gets its own id 1, 2, … (background 0) and the number of components. Use a breadth-first search (no SciPy).

In [ ]:
import collections
def label_components(mask):
    return None, None
mk = np.array([[1, 1, 0, 0, 1], [0, 1, 0, 1, 1], [0, 0, 0, 0, 0], [1, 0, 1, 1, 0], [1, 0, 0, 1, 0]])
lab_, n_ = label_components(mk)
check('number of components', n_, 4)
check('same blob, same id', None if lab_ is None else int(lab_[0, 0] == lab_[1, 1]), 1)
from scipy.ndimage import label as _sp_label
B = rng.random((30, 30)) < 0.45
check('matches scipy.ndimage.label count', label_components(B)[1], _sp_label(B)[1])

<details>
<summary><b>💡 Hint</b></summary>

Scan pixels; when you find an unlabelled foreground pixel, start a BFS with `collections.deque`, labelling all reachable 4-neighbours.

</details>

<details>
<summary><b>✅ Solution</b></summary>

This is the classic post-processing step for semantic masks (e.g. counting cells). It fails when objects touch, which is what instance segmentation (Mask R-CNN) or watershed on a distance transform fixes.

```python
import collections
def label_components(mask):
    lab = np.zeros(mask.shape, int); n = 0
    for start in zip(*np.nonzero(mask)):
        if lab[start]:
            continue
        n += 1; lab[start] = n; q = collections.deque([start])
        while q:
            r, c = q.popleft()
            for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                rr, cc = r + dr, c + dc
                if 0 <= rr < mask.shape[0] and 0 <= cc < mask.shape[1] and mask[rr, cc] and not lab[rr, cc]:
                    lab[rr, cc] = n; q.append((rr, cc))
    return lab, n
```

</details>

### Exercise 13 · Bilinear upsampling
*💻 Code · ★★☆*

Implement `upsample_bilinear(x, factor)` for a 2-D array using **align-corners** sampling: output pixel $(i, j)$ samples the input at $\big(i\frac{H-1}{H_{out}-1},\,j\frac{W-1}{W_{out}-1}\big)$ with $H_{out}=H\cdot$factor, interpolating linearly between the 4 nearest input pixels.

In [ ]:
def upsample_bilinear(x, factor):
    return None
x_ = np.array([[0., 1.], [2., 3.]]); u_ = upsample_bilinear(x_, 2)
check('shape', None if u_ is None else u_.shape, (4, 4))
check('corners preserved', None if u_ is None else u_[[0, 0, -1, -1], [0, -1, 0, -1]], [0, 1, 2, 3])
check('linear ramp stays linear', upsample_bilinear(np.tile(np.arange(4.), (3, 1)), 3), np.tile(np.linspace(0, 3, 12), (9, 1)))

<details>
<summary><b>💡 Hint</b></summary>

Compute fractional source coordinates, take `floor` (clipped so `y0+1` stays in range) and the fractional parts `wy, wx`, then blend the 4 neighbours.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Bilinear upsampling has no parameters and no checkerboard artefacts; decoders often use it followed by a 3×3 convolution instead of a transposed convolution.

```python
def upsample_bilinear(x, factor):
    H, W = x.shape; Ho, Wo = H * factor, W * factor
    ys = np.arange(Ho) * (H - 1) / (Ho - 1); xs = np.arange(Wo) * (W - 1) / (Wo - 1)
    y0 = np.clip(np.floor(ys).astype(int), 0, H - 2); x0 = np.clip(np.floor(xs).astype(int), 0, W - 2)
    wy = (ys - y0)[:, None]; wx = (xs - x0)[None, :]
    a, b = x[y0][:, x0], x[y0][:, x0 + 1]; c, d = x[y0 + 1][:, x0], x[y0 + 1][:, x0 + 1]
    return (1 - wy) * ((1 - wx) * a + wx * b) + wy * ((1 - wx) * c + wx * d)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Image Segmentation](Image%20Segmentation.md).*